# 01 | Penggabungan Berita & Kurs (menjadi `daily_merged.csv`)

**Input:**
- `data/processed/cnbc_articles_clean.csv`: Artikel berita hasil cleaning (kolom `published_wib`, `title_clean`, `body_clean`, dst.)
- `data/processed/jisdor_clean.csv`: Kurs USD/IDR bersih (kolom `tanggal`, `kurs_jisdor`)

**Output:**
- `data/interim/articles_aligned.csv` : Satu baris per **artikel**, sudah dipetakan ke trading day (`trading_date`) + `decay_weight`
- `data/processed/daily_merged.csv` : Satu baris per **trading day** (kalender penuh dari kurs), digabung dengan agregat berita hari tersebut.

Dipakai sebagai input `02_split_eda.ipynb` dan `03_feature_extraction.ipynb`.

### Prinsip
1. **Reuse** `get_effective_trading_date()` dari `src/jisdor/temporal_alignment.py` untuk memetakan tiap artikel ke trading day (bukan reimplementasi).
2. **Kalender kurs sebagai basis** (LEFT JOIN): Semua trading day dari `jisdor_clean.csv` tetap muncul walau tidak ada berita agar tidak bias ke hari yang kebetulan ada berita.
3. **Target dihitung dengan `shift(-1)`**: Fitur hari t dipasangkan ke label hari t+1, versi log-return serta arah biner.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").exists():
            return candidate
    raise RuntimeError(
        "Repo root tidak ditemukan. Jalankan notebook dari dalam repo "
        "nlp-trio-bertiga (folder manapun di dalamnya)."
    )

ROOT = _find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

from src.jisdor.temporal_alignment import get_effective_trading_date, get_indonesia_holidays, TIMEZONE_WIB

print(f"Repo root: {ROOT}")

Repo root: d:\Kuliah\Tugas dan Projek\Pemrosesan Bahasa Alami\repo\nlp-trio-bertiga


## 1. Muat & selaraskan artikel ke trading day

Tiap artikel dipetakan ke `trading_date` (tanggal kurs yang seharusnya menerima efek berita itu) memakai `get_effective_trading_date()`. Aturan jam tutup pasar 16:00 WIB, weekend, dan libur nasional sudah ditangani di fungsi.

In [2]:
arts = pd.read_csv(DATA / "processed" / "cnbc_articles_clean.csv", parse_dates=["published_wib"])
print(f"Artikel dimuat: {len(arts)} | {arts['published_wib'].min()} -> {arts['published_wib'].max()}")

# Holiday set mencakup seluruh rentang tahun berita + kurs (supaya konsisten dipakai lagi di cell kurs)
years = sorted(set(arts["published_wib"].dt.year) | set(range(2021, 2027)))
holiday_set = get_indonesia_holidays(years)
print(f"Hari libur nasional terkumpul: {len(holiday_set)} (tahun {years[0]}-{years[-1]})")

arts["trading_date"] = pd.to_datetime(
    arts["published_wib"].apply(lambda ts: get_effective_trading_date(ts, holiday_set, source_tz="WIB"))
)

n_shifted = (arts["trading_date"].dt.date != arts["published_wib"].dt.date).sum()
print(f"Artikel yang trading_date-nya bergeser dari tanggal publish: {n_shifted}/{len(arts)} "
      f"(after-hours / weekend / libur)")

Artikel dimuat: 136 | 2021-09-02 09:39:44+07:00 -> 2022-06-15 17:33:58+07:00
Hari libur nasional terkumpul: 100 (tahun 2021-2026)
Artikel yang trading_date-nya bergeser dari tanggal publish: 66/136 (after-hours / weekend / libur)


### `decay_weight` | Bobot time-decay tiap artikel

`hours_to_cutoff` = jarak waktu publish artikel ke jam tutup pasar (16:00 WIB) pada `trading_date` (selalu ≥ 0, karena `get_effective_trading_date()` menjamin cutoff ≥ waktu publish).  
Artikel yang publish lebih dekat ke cutoff diberi bobot lebih tinggi saat agregasi sentimen harian.  
Asumsi: berita yang lebih baru relatif terhadap penutupan hari itu lebih relevan dengan pergerakan kurs hari itu.

`decay_weight = exp(-hours_to_cutoff / 24)`Time constant 24 jam sebagai heuristik awal.

In [3]:
DECAY_TAU_HOURS = 24.0

cutoff = (arts["trading_date"] + pd.Timedelta(hours=16)).dt.tz_localize(TIMEZONE_WIB)
arts["hours_to_cutoff"] = (cutoff - arts["published_wib"]).dt.total_seconds() / 3600
assert (arts["hours_to_cutoff"] >= 0).all(), "hours_to_cutoff seharusnya selalu >= 0"
arts["decay_weight"] = np.exp(-arts["hours_to_cutoff"] / DECAY_TAU_HOURS)

print(arts[["published_wib", "trading_date", "hours_to_cutoff", "decay_weight"]].head(5))

              published_wib trading_date  hours_to_cutoff  decay_weight
0 2021-09-02 09:39:44+07:00   2021-09-02         6.337778      0.767917
1 2021-09-07 17:37:21+07:00   2021-09-08        22.377500      0.393610
2 2021-09-10 11:36:17+07:00   2021-09-10         4.395278      0.832654
3 2021-09-15 16:30:36+07:00   2021-09-16        23.490000      0.375781
4 2021-09-19 19:00:04+07:00   2021-09-20        20.998889      0.416881


In [4]:
# Simpan level artikel
articles_aligned = arts.rename(
    columns={"title_clean": "title", "body_clean": "body", "published_wib": "publish_wib"}
)[
    ["url", "trading_date", "publish_wib", "title", "body", "geo_hits", "hours_to_cutoff", "decay_weight"]
]

interim_dir = DATA / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)
articles_aligned.to_csv(interim_dir / "articles_aligned.csv", index=False)
print(f"Tersimpan: {interim_dir / 'articles_aligned.csv'} ({len(articles_aligned)} baris)")

Tersimpan: d:\Kuliah\Tugas dan Projek\Pemrosesan Bahasa Alami\repo\nlp-trio-bertiga\data\interim\articles_aligned.csv (136 baris)


## 2. Agregasi artikel dalam rentang harian
Satu baris per `trading_date`: jumlah artikel, gabungan teks (judul & isi dipisah), dan rata-rata `hours_to_cutoff`

In [5]:
daily_news = (
    articles_aligned.groupby("trading_date")
    .agg(
        n_articles=("url", "count"),
        text_title=("title", lambda s: " ".join(s)),
        text_body=("body", lambda s: " ".join(s)),
        mean_hours_to_cutoff=("hours_to_cutoff", "mean"),
    )
    .reset_index()
    .rename(columns={"trading_date": "date"})
)
daily_news["has_news"] = 1
print(f"Hari dengan berita: {len(daily_news)}")
daily_news.head(3)

Hari dengan berita: 85


,date,n_articles,text_title,text_body,mean_hours_to_cutoff,has_news
0,2021-09-02,1,China's health-care sector could be Beijing's ...,China's health-care sector will probably be th...,6.337778,1
1,2021-09-08,1,Guinea coup rattles iron ore and bauxite marke...,A military junta claimed to have seized contro...,22.377500,1
2,2021-09-10,1,"Biden, Xi discuss avoiding confrontation in se...",BEIJING - Chinese President Xi Jinping and US ...,4.395278,1


## 3. Kalender kurs + target

Basis kalender penuh dari `jisdor_clean.csv`. `log_ret`/`direction` = pergerakan hari itu; `ret_next`/`direction_next` = target (shift -1, prediksi besok).

In [6]:
kurs = (
    pd.read_csv(DATA / "processed" / "jisdor_clean.csv", parse_dates=["tanggal"])
    .sort_values("tanggal")
    .reset_index(drop=True)
    .rename(columns={"tanggal": "date", "kurs_jisdor": "rate"})
)

kurs["log_ret"] = np.log(kurs["rate"] / kurs["rate"].shift(1))
kurs["direction"] = (kurs["log_ret"] > 0).astype(float)
kurs["ret_next"] = kurs["log_ret"].shift(-1)
kurs["direction_next"] = kurs["direction"].shift(-1)
kurs["gap_days"] = kurs["date"].diff().dt.days
kurs["dayofweek"] = kurs["date"].dt.dayofweek

print(f"Kalender kurs: {len(kurs)} trading day, {kurs['date'].min().date()} -> {kurs['date'].max().date()}")

Kalender kurs: 1202 trading day, 2021-09-01 -> 2026-09-01


## 4. Join & simpan `daily_merged.csv`

LEFT JOIN kalender kurs dengan agregat berita {hari tanpa berita tetap ada (`has_news=0`, teks kosong)}. Baris tanpa target dibuang.

In [7]:
daily_merged = kurs.merge(daily_news, on="date", how="left")
daily_merged["has_news"] = daily_merged["has_news"].fillna(0).astype(int)
daily_merged["n_articles"] = daily_merged["n_articles"].fillna(0).astype(int)
daily_merged["text_title"] = daily_merged["text_title"].fillna("")
daily_merged["text_body"] = daily_merged["text_body"].fillna("")
daily_merged["mean_hours_to_cutoff"] = daily_merged["mean_hours_to_cutoff"].fillna(0.0)

before = len(daily_merged)
daily_merged = daily_merged.dropna(subset=["log_ret", "ret_next"]).reset_index(drop=True)
print(f"Baris dibuang (tanpa target log_ret/ret_next): {before - len(daily_merged)} | sisa: {len(daily_merged)}")
print(f"Hari dengan berita (setelah join ke kalender kurs): {daily_merged['has_news'].sum()}")
print(f"Rentang tanggal: {daily_merged['date'].min().date()} -> {daily_merged['date'].max().date()}")

daily_merged.head(3)

Baris dibuang (tanpa target log_ret/ret_next): 2 | sisa: 1200
Hari dengan berita (setelah join ke kalender kurs): 82
Rentang tanggal: 2021-09-02 -> 2026-08-31


,date,rate,log_ret,direction,ret_next,direction_next,gap_days,dayofweek,n_articles,text_title,text_body,mean_hours_to_cutoff,has_news
0,2021-09-02,14281,-0.000210,0.0,-0.001401,0.0,1.0,3,1,China's health-care sector could be Beijing's ...,China's health-care sector will probably be th...,6.337778,1
1,2021-09-03,14261,-0.001401,0.0,-0.001544,0.0,1.0,4,0,,,0.000000,0
2,2021-09-06,14239,-0.001544,0.0,-0.003095,0.0,3.0,0,0,,,0.000000,0


In [8]:
processed_dir = DATA / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
daily_merged.to_csv(processed_dir / "daily_merged.csv", index=False)
print(f"Tersimpan: {processed_dir / 'daily_merged.csv'}")

Tersimpan: d:\Kuliah\Tugas dan Projek\Pemrosesan Bahasa Alami\repo\nlp-trio-bertiga\data\processed\daily_merged.csv


## 5. Sanity check kolom untuk notebook 02 & 03

In [9]:
required_daily = [
    "date", "rate", "log_ret", "ret_next", "direction", "direction_next",
    "has_news", "n_articles", "text_title", "text_body", "gap_days",
    "dayofweek", "mean_hours_to_cutoff",
]
missing_daily = [c for c in required_daily if c not in daily_merged.columns]
assert not missing_daily, f"Kolom hilang di daily_merged.csv: {missing_daily}"

required_arts = ["trading_date", "publish_wib", "title", "body", "decay_weight"]
missing_arts = [c for c in required_arts if c not in articles_aligned.columns]
assert not missing_arts, f"Kolom hilang di articles_aligned.csv: {missing_arts}"

print("OK,  semua kolom yang dibutuhkan 02_split_eda.ipynb dan 03_feature_extraction.ipynb tersedia.")

OK,  semua kolom yang dibutuhkan 02_split_eda.ipynb dan 03_feature_extraction.ipynb tersedia.
